# 第 55 课：推理任务与可验证评分器

**回到模型能力：在增加采样、搜索或推理预算之前，先把“怎样算对”写清楚。**

上一课完成了令牌生命周期；本课不再处理登录或工具授权，而是构建数学与代码任务的评分底座，为后续多次采样、候选选择与测试时计算做准备。

每课两个文件：本 Notebook 内置完整实现，不依赖同名 `.py`；完整脚本也可独立运行。

**环境：Python 3.10+，仅标准库，CPU 即可。默认不联网、不下载、不调用模型、不执行任意 Python。**

下方的成绩来自手工构造的响应与参考代码，是评分器机制实验，不是语言模型能力结果。

## 本课完成标准

学完后你应能解释并验证：

1. 为什么子串匹配、随意去掉负号、自动找最后一个数字会误判。
2. 怎样区分精确值、声明的近似容差和物理单位。
3. 为什么有多个合法解时要验证约束，而不是只匹配一份答案。
4. 怎样区分公开样例、额外评测测试和对所有输入正确的形式证明。
5. 为什么先审计评分器，才能放心地用它筛选更多模型候选。

建议分两次：第一次完成数值、单位、见证；第二次完成代码、批量评测、练习。无需背诵文件读写和命令行实现。

## 0．术语与边界

| 名称 | 在本课中的含义 |
|---|---|
| Generator | 给出候选答案或代码。本课用手工响应替代，不进行模型推理。 |
| Grader / verifier | 对照任务契约进行检查，输出分项结果。不是万能真理判定器。 |
| 参考答案／评测测试 | 宿主拥有的评测材料，不进入模型请求。 |
| 见证 witness | 一个满足约束的可检查对象，例如一个不重复整数集合。不是自然语言证明。 |
| Outcome verification | 检查提交结果，不认证模型内部推理过程。 |

用验证器给多个候选排序，是推理研究中的一条路线；已有研究训练了专门的数学答案验证器。但**本课只实现确定性的任务评分器，不训练奖励模型，也不是 GSM8K、MATH 或 HumanEval 的复刻**。[Training Verifiers](https://arxiv.org/abs/2110.14168)

所有教学答案与测试代码公开。“留出”只表示不传入生成请求，不能当作秘密评测服务器或防污染保证。

In [1]:
import sys
from pathlib import Path
print("Python：", sys.version.split()[0])
print("运行目录：", Path.cwd())
print("本课仅标准库；不调用模型；不直接执行候选源码。")

Python： 3.13.5
运行目录： /mnt/data/_lesson55_validation/notebook
本课仅标准库；不调用模型；不直接执行候选源码。


## 工程底座：先运行下面十个代码单元

这些单元按顺序定义完整实现，默认折叠源代码；Jupyter 不支持折叠时仍可正常运行。

| 单元 | 内容 |
|---|---|
| 1–2 | 严格 JSON、精确有理数、单位与见证 |
| 3 | 有限 AST 解释器 |
| 4–6 | 固定题目、请求隔离、评分、手工响应与机制实验 |
| 7–9 | 练习检查、参考实现、固定检查与故障注入 |
| 10 | 导出、重评分、独立 CLI |

候选只作为字符串解析为 AST，再由有限解释器读取允许节点。**不要把本课代码改成 `exec(candidate)` 来扩展语言能力。** 官方 HumanEval 执行代码也提醒，常规限制不是安全沙箱，不能盲目执行不可信代码。[HumanEval 执行说明](https://github.com/openai/human-eval/blob/master/human_eval/execution.py)

教学解释器仍不是经过安全认证的隔离系统；解析器和运行环境本身也可能出错。

In [2]:
from __future__ import annotations

import argparse
import ast
import copy
import hashlib
import itertools
import json
import math
import platform
import re
import sys
from collections import Counter
from contextlib import contextmanager
from datetime import datetime, timezone
from fractions import Fraction
from pathlib import Path

VERSION = "lesson55-verifiable-graders-v1"
MAX_RESPONSE_BYTES = 16384
MAX_SOURCE_BYTES = 4096
MAX_AST_NODES = 300
MAX_AST_DEPTH = 24
MAX_STEPS = 2000
MAX_INTEGER = 10**9
MAX_INPUT_LEN = 16
MAX_FILE_BYTES = 4 * 1024 * 1024
MAX_RECORDS = 256
NUMBER_RE = re.compile(r"[+-]?[0-9]+(?:/[0-9]+|\.[0-9]+)?\Z")
SOURCES = {
    "fractions": "https://docs.python.org/3.13/library/fractions.html",
    "json": "https://docs.python.org/3.13/library/json.html",
    "ast": "https://docs.python.org/3.13/library/ast.html",
    "verifiers": "https://arxiv.org/abs/2110.14168",
    "process_outcome": "https://arxiv.org/abs/2305.20050",
    "code_warning": "https://github.com/openai/human-eval/blob/master/human_eval/execution.py",
}


class LessonError(ValueError):
    """有界输入或协议不满足本课约定。"""


class UnsupportedCode(LessonError):
    """超出课堂语言范围，不等于该代码在完整 Python 中语义错误。"""


class RuntimeLimit(LessonError):
    """有限解释器的运行预算耗尽。"""


def require(ok: bool, message: str) -> None:
    if not ok:
        raise LessonError(message)


def canonical(value) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def bounded_text(value, limit: int, label: str = "text") -> str:
    require(type(value) is str, f"{label} 必须是字符串")
    try:
        size = len(value.encode("utf-8", errors="strict"))
    except UnicodeError as exc:
        raise LessonError(f"{label} 包含非法 Unicode") from exc
    require(size <= limit, f"{label} 超长")
    return value


def strict_json(text: str):
    bounded_text(text, MAX_RESPONSE_BYTES, "JSON")
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, "重复 JSON 键")
            result[key] = value
        return result
    def invalid_constant(_):
        raise LessonError("不允许 NaN/Infinity")
    def small_int(raw):
        require(len(raw.lstrip("-")) <= 24, "JSON 整数过长")
        return int(raw)
    try:
        value = json.loads(text, object_pairs_hook=pairs,
                           parse_constant=invalid_constant, parse_int=small_int)
    except (json.JSONDecodeError, RecursionError, ValueError) as exc:
        if isinstance(exc, LessonError):
            raise
        raise LessonError("非法或过深 JSON") from exc
    stack = [(value, 0)]
    seen = 0
    while stack:
        node, depth = stack.pop()
        seen += 1
        require(depth <= 16 and seen <= 2048, "JSON 结构过大")
        if type(node) is float:
            require(math.isfinite(node), "JSON 非有限数值")
        elif type(node) is str:
            bounded_text(node, MAX_RESPONSE_BYTES, "JSON 字符串")
        elif type(node) is list:
            stack.extend((item, depth + 1) for item in node)
        elif type(node) is dict:
            stack.extend((item, depth + 1) for item in node.values())
    return value


def exact_keys(value, keys: set[str]) -> None:
    require(type(value) is dict and set(value) == keys, "字段集合不符合本题约定")


def parse_rational(value: str) -> Fraction:
    """整数、小数、分数的精确解析。不接受指数、百分号、多个答案或 float。"""
    text = bounded_text(value, 64, "answer").strip()
    require(NUMBER_RE.fullmatch(text) is not None, "需要一个有符号数值字符串")
    require(sum(c.isdigit() for c in text) <= 24, "数值有效位过多")
    try:
        result = Fraction(text)  # 不能先调用 float()
    except (ValueError, ZeroDivisionError) as exc:
        raise LessonError("无效分数或零分母") from exc
    return result


def close_rational(actual: str, expected: str, *, atol: str, rtol: str) -> bool:
    """独立的近似任务示例；不用在默认精确题上。规则：绝对项+相对项。"""
    a, e = parse_rational(actual), parse_rational(expected)
    absolute, relative = parse_rational(atol), parse_rational(rtol)
    require(absolute >= 0 and relative >= 0, "容差不能为负")
    return abs(a - e) <= absolute + relative * abs(e)

In [3]:
# (物理量类别, 换算成基准单位的精确倍数)
UNITS = {
    "s": ("time", "1"), "秒": ("time", "1"),
    "min": ("time", "60"), "minute": ("time", "60"), "分钟": ("time", "60"),
    "h": ("time", "3600"), "小时": ("time", "3600"),
    "m": ("length", "1"), "米": ("length", "1"),
    "cm": ("length", "1/100"), "厘米": ("length", "1/100"),
    "mm": ("length", "1/1000"),
    "CNY": ("money", "1"), "元": ("money", "1"), "fen": ("money", "1/100"),
}


def normalize_quantity(answer: str, unit: str) -> tuple[str, Fraction]:
    bounded_text(unit, 32, "unit")
    unit = unit.strip()
    require(unit in UNITS, "单位不在本课显式换算表中")
    dimension, scale = UNITS[unit]
    return dimension, parse_rational(answer) * Fraction(scale)


def verify_exact(task: dict, payload: dict) -> dict:
    exact_keys(payload, {"answer"})
    actual, expected = parse_rational(payload["answer"]), Fraction(task["expected"])
    return {"passed": actual == expected, "canonical_answer": str(actual),
            "reason": "equal" if actual == expected else "wrong_value",
            "scope": "final_exact_value_not_reasoning_process"}


def verify_quantity(task: dict, payload: dict) -> dict:
    exact_keys(payload, {"answer", "unit"})
    dim, val = normalize_quantity(payload["answer"], payload["unit"])
    target_dim, target_val = normalize_quantity(task["expected"], task["unit"])
    passed = dim == target_dim and val == target_val
    return {"passed": passed, "dimension": dim, "canonical_value": str(val),
            "reason": "equal_quantity" if passed else
                      ("wrong_dimension" if dim != target_dim else "wrong_quantity"),
            "scope": "quantity_with_explicit_unit_table"}


def verify_witness(task: dict, payload: dict) -> dict:
    """检查构造是否满足全部约束，不强求等于某一个参考解。"""
    exact_keys(payload, {"items"})
    items = payload["items"]
    require(type(items) is list and len(items) <= 16, "items 必须是有界列表")
    require(all(type(x) is int and abs(x) <= MAX_INTEGER for x in items),
            "items 只允许有界整数；bool 不算整数答案")
    checks = {
        "count": len(items) == task["count"],
        "membership": all(x in task["allowed"] for x in items),
        "distinct": len(set(items)) == len(items),
        "sum": sum(items) == task["target_sum"],
    }
    return {"passed": all(checks.values()), "checks": checks,
            "canonical_items": sorted(items), "scope": "all_stated_finite_constraints"}

In [4]:
BIN_OPS = {ast.Add, ast.Sub, ast.Mult, ast.FloorDiv, ast.Mod}
CMP_OPS = {ast.Eq, ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE}
ALLOWED_AST = {
    ast.Module, ast.FunctionDef, ast.arguments, ast.arg, ast.Assign, ast.AugAssign,
    ast.For, ast.If, ast.Return, ast.Name, ast.Load, ast.Store, ast.Constant,
    ast.BinOp, ast.UnaryOp, ast.UAdd, ast.USub, ast.Not, ast.Compare,
    ast.BoolOp, ast.And, ast.Or,
} | BIN_OPS | CMP_OPS


def int_value(value) -> int:
    require(type(value) is int and abs(value) <= MAX_INTEGER, "需要有界整数，不能使用bool")
    return value


def local_name(name: str) -> None:
    require(re.fullmatch(r"[a-z][a-z0-9_]{0,23}", name) is not None
            and "__" not in name and name not in {"solve"}, "非法局部变量名")


def parse_program(source: str) -> ast.FunctionDef:
    try:
        bounded_text(source, MAX_SOURCE_BYTES, "source")
        tree = ast.parse(source, mode="exec")  # 仅构建语法树，不编译执行
        require(len(tree.body) == 1 and type(tree.body[0]) is ast.FunctionDef,
                "只允许一个 solve(numbers) 函数")
        func = tree.body[0]
        args = func.args
        require(func.name == "solve" and not func.decorator_list and func.returns is None
                and not getattr(func, "type_params", []) and func.type_comment is None,
                "函数头不符合约定")
        require(len(args.args) == 1 and args.args[0].arg == "numbers"
                and args.args[0].annotation is None and not args.posonlyargs
                and not args.kwonlyargs and not args.defaults and not args.kw_defaults
                and args.vararg is None and args.kwarg is None, "参数必须只有 numbers")
        stack, total, names = [(tree, 0)], 0, set()
        while stack:
            node, depth = stack.pop()
            total += 1
            require(total <= MAX_AST_NODES and depth <= MAX_AST_DEPTH, "AST过大或过深")
            require(type(node) in ALLOWED_AST, f"不支持的语法：{type(node).__name__}")
            if isinstance(node, ast.FunctionDef):
                require(node is func, "不允许嵌套函数")
            if isinstance(node, ast.Constant):
                int_value(node.value)
            if isinstance(node, ast.Name):
                local_name(node.id)
                if isinstance(node.ctx, ast.Store):
                    require(node.id != "numbers", "不能改写输入")
                    names.add(node.id)
                    require(len(names) <= 16, "局部变量太多")
            if isinstance(node, ast.Assign):
                require(len(node.targets) == 1 and isinstance(node.targets[0], ast.Name),
                        "只支持一个局部变量赋值")
            if isinstance(node, ast.AugAssign):
                require(isinstance(node.target, ast.Name) and type(node.op) in BIN_OPS,
                        "不支持的增量赋值")
            if isinstance(node, ast.For):
                require(isinstance(node.target, ast.Name)
                        and isinstance(node.iter, ast.Name) and node.iter.id == "numbers"
                        and not node.orelse, "for 只能遍历原始 numbers，无 else")
            if isinstance(node, ast.Compare):
                require(len(node.ops) == len(node.comparators) == 1,
                        "比较必须是单一比较")
            if isinstance(node, ast.Return):
                require(node.value is not None, "需要返回整数")
            stack.extend((child, depth + 1) for child in ast.iter_child_nodes(node))
        return func
    except (SyntaxError, RecursionError, ValueError) as exc:
        raise UnsupportedCode(str(exc)[:180]) from exc


class _Returned(Exception):
    def __init__(self, value):
        self.value = int_value(value)


class TinyRunner:
    """解释整数语言。没有 eval/exec；有限输入与步骤不是通用OS沙箱。"""
    def __init__(self, program: ast.FunctionDef, step_limit: int = MAX_STEPS):
        require(type(step_limit) is int and 0 < step_limit <= 10000, "非法步骤上限")
        self.program, self.limit = program, step_limit
        self.used, self.env, self.numbers = 0, {}, ()

    def tick(self):
        self.used += 1
        if self.used > self.limit:
            raise RuntimeLimit("step_budget_exhausted")

    def binary(self, op, left, right):
        a, b = int_value(left), int_value(right)
        if isinstance(op, ast.Add): value = a + b
        elif isinstance(op, ast.Sub): value = a - b
        elif isinstance(op, ast.Mult): value = a * b
        elif isinstance(op, ast.FloorDiv):
            require(b != 0, "division_by_zero")
            value = a // b
        elif isinstance(op, ast.Mod):
            require(b != 0, "division_by_zero")
            value = a % b
        else: raise LessonError("unknown_operator")
        return int_value(value)

    def expr(self, node):
        self.tick()
        if isinstance(node, ast.Constant): return int_value(node.value)
        if isinstance(node, ast.Name):
            require(node.id != "numbers" and node.id in self.env, "uninitialized_or_non_scalar")
            return self.env[node.id]
        if isinstance(node, ast.BinOp):
            return self.binary(node.op, self.expr(node.left), self.expr(node.right))
        if isinstance(node, ast.UnaryOp):
            val = self.expr(node.operand)
            if isinstance(node.op, ast.Not):
                require(type(val) is bool, "not 仅用于布尔条件")
                return not val
            val = int_value(val)
            return val if isinstance(node.op, ast.UAdd) else -val
        if isinstance(node, ast.Compare):
            a, b = int_value(self.expr(node.left)), int_value(self.expr(node.comparators[0]))
            op = node.ops[0]
            if isinstance(op, ast.Eq): return a == b
            if isinstance(op, ast.NotEq): return a != b
            if isinstance(op, ast.Lt): return a < b
            if isinstance(op, ast.LtE): return a <= b
            if isinstance(op, ast.Gt): return a > b
            return a >= b
        if isinstance(node, ast.BoolOp):
            for child in node.values:
                val = self.expr(child)
                require(type(val) is bool, "and/or 仅用于布尔条件")
                if isinstance(node.op, ast.And) and not val: return False
                if isinstance(node.op, ast.Or) and val: return True
            return isinstance(node.op, ast.And)
        raise LessonError("unsupported_expression")

    def statements(self, body):
        for node in body:
            self.tick()
            if isinstance(node, ast.Assign):
                self.env[node.targets[0].id] = int_value(self.expr(node.value))
            elif isinstance(node, ast.AugAssign):
                name = node.target.id
                require(name in self.env, "uninitialized_variable")
                self.env[name] = self.binary(node.op, self.env[name], self.expr(node.value))
            elif isinstance(node, ast.If):
                cond = self.expr(node.test)
                require(type(cond) is bool, "if 需要比较条件")
                self.statements(node.body if cond else node.orelse)
            elif isinstance(node, ast.For):
                for item in self.numbers:
                    self.tick()
                    self.env[node.target.id] = item
                    self.statements(node.body)
            elif isinstance(node, ast.Return):
                raise _Returned(self.expr(node.value))
            else:
                raise LessonError("unsupported_statement")

    def run(self, numbers: list[int]) -> int:
        require(type(numbers) is list and len(numbers) <= MAX_INPUT_LEN, "输入必须是短整数列表")
        require(all(type(x) is int and abs(x) <= 1000 for x in numbers), "输入整数范围错误")
        self.used, self.env, self.numbers = 0, {}, tuple(numbers)
        try:
            self.statements(self.program.body)
        except _Returned as done:
            return done.value
        raise LessonError("missing_return")


def run_source(source: str, numbers: list[int], step_limit: int = MAX_STEPS) -> int:
    return TinyRunner(parse_program(source), step_limit).run(numbers)

In [5]:
CORRECT_CODE = {
    "sum_abs": """def solve(numbers):
    total = 0
    for value in numbers:
        if value < 0:
            total = total - value
        else:
            total = total + value
    return total
""",
    "count_nonzero": """def solve(numbers):
    count = 0
    for value in numbers:
        if value != 0:
            count = count + 1
    return count
""",
    "max_or_zero": """def solve(numbers):
    best = 0
    seen = 0
    for value in numbers:
        if seen == 0 or value > best:
            best = value
        seen = 1
    return best
""",
    "sum_nonnegative": """def solve(numbers):
    total = 0
    for value in numbers:
        if value >= 0:
            total = total + value
    return total
""",
}
WRONG_CODE = {
    "count_nonzero": CORRECT_CODE["count_nonzero"].replace("value != 0", "value > 0"),
    "max_or_zero": CORRECT_CODE["max_or_zero"].replace("seen == 0 or value > best", "value > best"),
    "sum_nonnegative": CORRECT_CODE["sum_nonnegative"].replace("value >= 0", "value >= 0 and value < 10"),
}
LANGUAGE_RULE = (
    "仅输出JSON {\"code\":\"...\"}。代码仅允许 def solve(numbers)，返回整数。"
    "输入是长度0至16、元素为[-1000,1000]整数的列表。只允许整数局部变量、"
    "+ - * // %、单一比较、and/or/not、if/else、for value in numbers、return；"
    "禁止函数调用(包括sum/len)、导入、属性、索引、while、递归、字符串、列表构造。"
    "源码最多4096个UTF-8字节；每例2000个解释步骤，整数绝对值不超过10^9。"
    "不要求解释推理，不使用代码围栏。"
)
NUMBER_RULE = (
    "仅输出JSON {\"answer\":\"数值\"}；数值是整数、普通十进制小数或分数的字符串，"
    "合计最多24个数字；保留正负号，不用指数、百分号、单位、解释或代码围栏。精确比较，不四舍五入。"
)
QUANTITY_RULE = (
    "仅输出JSON {\"answer\":\"数值\",\"unit\":\"单位\"}。数值用字符串；"
    "允许精确等价的单位换算。单位表：s/秒、min/minute/分钟、h/小时；"
    "m/米、cm/厘米、mm；CNY/元、fen(分)。不允许无单位，也不能混淆物理量。"
)
WITNESS_RULE = "仅输出JSON {\"items\":[整数,...]}。顺序不限，元素必须互不重复。不要输出理由。"


def trusted_oracle(name: str, numbers: list[int]) -> int:
    """宿主代码，不是执行候选。与候选解释器走不同计算路径。"""
    if name == "sum_abs": return sum(map(abs, numbers))
    if name == "count_nonzero": return len([x for x in numbers if x != 0])
    if name == "max_or_zero": return max(numbers, default=0)
    if name == "sum_nonnegative": return sum(x for x in numbers if x >= 0)
    raise LessonError("未知可信参考函数")


def code_cases(name: str) -> tuple[list[dict], list[dict]]:
    public_inputs = {
        "sum_abs": [[], [-1, 2], [0]],
        "count_nonzero": [[], [0, 0], [1, 0, 2]],
        "max_or_zero": [[], [1, 3, 2], [0]],
        "sum_nonnegative": [[], [1, 2], [-1, 2]],
    }[name]
    public = [{"input": xs, "expected": trusted_oracle(name, xs)} for xs in public_inputs]
    pool = [list(xs) for n in range(4) for xs in itertools.product([-2, -1, 0, 1, 2], repeat=n)]
    pool += [[10, -5, 20], [-12, -3], [1000, -1000, 0], [0] * 16,
             [-1] * 16, [10] * 16, [1000], [-1000]]
    seen = {tuple(xs) for xs in public_inputs}
    heldout = []
    for xs in pool:
        if tuple(xs) not in seen:
            seen.add(tuple(xs))
            heldout.append({"input": xs, "expected": trusted_oracle(name, xs)})
    return public, heldout


def make_tasks() -> list[dict]:
    tasks = []
    def add_numeric(id_, split, problem, expected):
        tasks.append({"id": id_, "split": split, "kind": "exact",
                      "problem": problem, "response_rule": NUMBER_RULE, "expected": expected})
    add_numeric("dev_fraction", "dev", "计算 3/4 + 1/8。", "7/8")
    add_numeric("fraction_sum", "test", "计算 1/3 + 1/6。", "1/2")
    add_numeric("signed_arithmetic", "test", "按通常运算顺序计算 -7 + 2 * 3。", "-1")
    add_numeric("decimal_sum", "test", "用精确十进制含义计算 0.1 + 0.2。", "3/10")
    for id_, split, problem, expected, unit in [
        ("dev_duration", "dev", "两小时等于多少分钟？", "120", "min"),
        ("duration", "test", "1.5小时等于多长时间？可用分钟作答。", "90", "min"),
        ("length", "test", "3/4米等于多长？可用厘米作答。", "75", "cm"),
        ("price", "test", "教学商店每杯茶2.50元，3杯总价多少？不计其他费用。", "7.5", "CNY"),
    ]:
        tasks.append({"id": id_, "split": split, "kind": "quantity", "problem": problem,
                      "expected": expected, "unit": unit, "response_rule": QUANTITY_RULE})
    for id_, split, allowed, count, total in [
        ("dev_subset", "dev", [1, 2, 3, 4, 5], 2, 6),
        ("subset_positive", "test", list(range(1, 9)), 3, 12),
        ("subset_signed", "test", [-3, -2, 0, 1, 4, 5], 2, 2),
        ("subset_primes", "test", [2, 3, 5, 7, 11, 13], 3, 21),
    ]:
        tasks.append({"id": id_, "split": split, "kind": "witness", "allowed": allowed,
                      "count": count, "target_sum": total,
                      "problem": f"从{allowed}选择{count}个互不相同的元素，使其和为{total}。给出任意合法选择。",
                      "response_rule": WITNESS_RULE})
    descriptions = {
        "sum_abs": "返回输入整数绝对值之和；空列表返回0。",
        "count_nonzero": "返回非零整数的数量；负数也算非零；空列表返回0。",
        "max_or_zero": "返回最大整数；只有空列表才返回0；全负列表不能错误返回0。",
        "sum_nonnegative": "返回所有非负整数之和；空列表返回0。",
    }
    for name, problem in descriptions.items():
        public, evaluation = code_cases(name)
        tasks.append({"id": "code_" + name, "split": "dev" if name == "sum_abs" else "test",
                      "kind": "code", "problem": problem, "oracle": name,
                      "public_cases": public, "evaluation_cases": evaluation,
                      "response_rule": LANGUAGE_RULE})
    return tasks


def public_request(task: dict) -> dict:
    """只白名单复制生成侧需要的字段；不导出答案、参考函数或评测测试。"""
    result = {"task_id": task["id"], "sample_id": "s0", "kind": task["kind"],
              "problem": task["problem"], "response_rule": task["response_rule"]}
    if task["kind"] == "code":
        result["public_examples"] = copy.deepcopy(task["public_cases"])
    return result


def make_plan(split: str = "test") -> dict:
    require(type(split) is str and split in {"dev", "test"}, "split 必须是dev或test")
    tasks = [t for t in make_tasks() if t["split"] == split]
    return {"version": VERSION, "split": split, "dataset_sha256": digest(tasks),
            "grader_contract_sha256": digest(grader_contract()),
            "slots": [{"task_id": t["id"], "sample_id": "s0",
                       "request_sha256": digest(public_request(t))} for t in tasks]}


def grader_contract() -> dict:
    return {"version": VERSION, "math": "exact_fraction_no_heuristic_extraction",
            "quantity": UNITS, "witness": "count_membership_distinct_sum",
            "code": "one_function_AST_interpreter_public_and_finite_evaluation_v1",
            "max_source_bytes": MAX_SOURCE_BYTES, "max_ast_nodes": MAX_AST_NODES,
            "max_ast_depth": MAX_AST_DEPTH, "max_steps": MAX_STEPS,
            "max_integer": MAX_INTEGER, "max_input_length": MAX_INPUT_LEN,
            "completion": "only_status_ok_and_eos_are_scored",
            "aggregation": "planned_slots_fixed_denominator_no_best_of",
            "limits": "not_a_formal_proof_not_a_general_python_sandbox_not_a_process_reward_model"}

In [6]:
def run_cases(program: ast.FunctionDef, cases: list[dict]) -> dict:
    rows = []
    for case in cases:
        error, actual, used = None, None, 0
        runner = TinyRunner(program)
        try:
            actual = runner.run(copy.deepcopy(case["input"]))
        except LessonError as exc:
            error = str(exc)
        used = runner.used
        rows.append({"input": case["input"], "expected": case["expected"],
                     "actual": actual, "error": error, "steps": used,
                     "passed": error is None and type(actual) is int and actual == case["expected"]})
    return {"passed": all(row["passed"] for row in rows),
            "passed_count": sum(row["passed"] for row in rows), "total": len(rows), "cases": rows}


def verify_code(task: dict, payload: dict) -> dict:
    exact_keys(payload, {"code"})
    program = parse_program(payload["code"])
    public = run_cases(program, task["public_cases"])
    evaluation = run_cases(program, task["evaluation_cases"])
    return {"passed": public["passed"] and evaluation["passed"],
            "public": public, "evaluation": evaluation,
            "scope": "finite_tests_not_all_inputs_not_code_security",
            "source_sha256": hashlib.sha256(payload["code"].encode()).hexdigest()}


def validate_record(record: dict) -> None:
    exact_keys(record, {"task_id", "sample_id", "status", "stop_reason", "output"})
    for key in ["task_id", "sample_id"]:
        value = bounded_text(record[key], 80, key)
        require(re.fullmatch(r"[A-Za-z0-9_-]+", value) is not None, "非法记录ID")
    require(type(record["status"]) is str and record["status"] in {"ok", "error"}, "status 非法")
    require(type(record["stop_reason"]) is str and record["stop_reason"] in {"eos", "length", "error"}, "stop_reason 非法")
    bounded_text(record["output"], MAX_RESPONSE_BYTES, "output")
    if record["status"] == "error":
        require(record["stop_reason"] == "error", "运行失败必须用error停止原因")
    else:
        require(record["stop_reason"] != "error", "状态与停止原因矛盾")


def grade_record(task: dict, record: dict | None) -> dict:
    row = {"task_id": task["id"], "sample_id": "s0", "kind": task["kind"],
           "format_ok": None, "answer_ok": None, "strict_success": False,
           "stage": "missing", "details": None, "raw_sha256": None}
    if record is None:
        return row
    validate_record(record)
    require(record["task_id"] == task["id"] and record["sample_id"] == "s0", "题目身份不匹配")
    row["raw_sha256"] = digest(record)
    if record["status"] == "error":
        row["stage"] = "generation_error"
        return row
    if record["stop_reason"] != "eos":
        row["stage"] = "incomplete_generation"
        return row  # 不能执行可能只是一部分的代码；也不默认修补答案
    try:
        payload = strict_json(record["output"])
        if task["kind"] == "exact": detail = verify_exact(task, payload)
        elif task["kind"] == "quantity": detail = verify_quantity(task, payload)
        elif task["kind"] == "witness": detail = verify_witness(task, payload)
        elif task["kind"] == "code": detail = verify_code(task, payload)
        else: raise LessonError("unknown_task_kind")
    except UnsupportedCode as exc:
        row.update(format_ok=False, stage="unsupported_code", details={"error": str(exc)})
        return row
    except LessonError as exc:
        row.update(format_ok=False, stage="invalid_format", details={"error": str(exc)})
        return row
    row.update(format_ok=True, answer_ok=detail["passed"], strict_success=detail["passed"],
               stage="passed" if detail["passed"] else "wrong_result", details=detail)
    return row


def summarize(rows: list[dict]) -> dict:
    require(len(rows) > 0, "没有计划题目")
    require(len({(r["task_id"], r["sample_id"]) for r in rows}) == len(rows), "重复评分位置")
    total = len(rows)
    strict = sum(row["strict_success"] is True for row in rows)
    categories = {}
    for kind in sorted({row["kind"] for row in rows}):
        subset = [row for row in rows if row["kind"] == kind]
        good = sum(row["strict_success"] is True for row in subset)
        categories[kind] = {"planned": len(subset), "passed": good, "rate": good / len(subset)}
    return {"planned": total, "received": sum(r["stage"] != "missing" for r in rows),
            "format_passed": sum(r["format_ok"] is True for r in rows),
            "verified_passed": strict, "success_rate": strict / total,
            "category_macro_rate": sum(v["rate"] for v in categories.values()) / len(categories),
            "stages": dict(sorted(Counter(r["stage"] for r in rows).items())),
            "categories": categories,
            "note": "缺失/运行错误计入计划分母，但不宣称它们已经被证明答案错误。"}


def score_records(records: list[dict], split: str = "test", plan: dict | None = None) -> dict:
    expected_plan = make_plan(split)
    if plan is not None:
        require(plan == expected_plan, "计划、题集或评分契约发生变化")
    require(type(records) is list and len(records) <= MAX_RECORDS, "响应记录过多")
    slots = {(s["task_id"], s["sample_id"]) for s in expected_plan["slots"]}
    by_key = {}
    for record in records:
        validate_record(record)
        key = (record["task_id"], record["sample_id"])
        require(key in slots, "不在本次计划中的响应")
        require(key not in by_key, "同一计划位置有多个响应，不允许暗中择优")
        by_key[key] = record
    rows = [grade_record(task, by_key.get((task["id"], "s0")))
            for task in make_tasks() if task["split"] == split]
    return {"version": VERSION, "plan": expected_plan, "rows": rows, "summary": summarize(rows),
            "evaluation_scope": "exact_values_units_constructive_constraints_and_finite_code_tests",
            "reasoning_process_verified": False, "model_identity_verified": False}

In [7]:
def candidate(task_id: str, payload: dict | str, *, status="ok", stop_reason="eos") -> dict:
    return {"task_id": task_id, "sample_id": "s0", "status": status,
            "stop_reason": stop_reason,
            "output": payload if isinstance(payload, str) else canonical(payload)}


def correct_payload(task: dict) -> dict:
    if task["kind"] == "exact": return {"answer": task["expected"]}
    if task["kind"] == "quantity": return {"answer": task["expected"], "unit": task["unit"]}
    if task["kind"] == "witness":
        solution = next(items for items in itertools.combinations(task["allowed"], task["count"])
                        if sum(items) == task["target_sum"])
        return {"items": list(solution)}
    return {"code": CORRECT_CODE[task["oracle"]]}


def fixture_records(name: str, split: str = "test") -> list[dict]:
    require(name in {"reference", "public_overfit", "mixed"}, "未知夹具")
    tasks = [task for task in make_tasks() if task["split"] == split]
    rows = [candidate(t["id"], correct_payload(t)) for t in tasks]
    if name == "reference": return rows
    if name == "public_overfit":
        for task, row in zip(tasks, rows):
            if task["kind"] == "code" and task["oracle"] in WRONG_CODE:
                row["output"] = canonical({"code": WRONG_CODE[task["oracle"]]})
        return rows
    require(split == "test", "mixed 夹具仅用于test")
    replacements = {
        "fraction_sum": candidate("fraction_sum", {"answer": "2/4"}),
        "signed_arithmetic": candidate("signed_arithmetic", {"answer": "1"}),
        "decimal_sum": candidate("decimal_sum", {"answer": "0.30000000000000004"}),
        "duration": candidate("duration", {"answer": "1.5", "unit": "h"}),
        "length": candidate("length", {"answer": "75", "unit": "m"}),
        "price": candidate("price", {"answer": "750", "unit": "fen"}),
        "subset_positive": candidate("subset_positive", {"items": [7, 4, 1]}),
        "subset_signed": candidate("subset_signed", {"items": [-3, 5]}, stop_reason="length"),
        "code_count_nonzero": candidate("code_count_nonzero", {"code": WRONG_CODE["count_nonzero"]}),
        "code_sum_nonnegative": candidate("code_sum_nonnegative", "", status="error", stop_reason="error"),
    }
    return [replacements.get(r["task_id"], r) for r in rows if r["task_id"] != "subset_primes"]


def metamorphic_report(source: str) -> dict:
    """性质检查是补充，不把性质成立当作唯一正确性证明。"""
    xs = [-2, 0, 3]
    base = run_source(source, xs)
    checks = {
        "reverse_invariant": run_source(source, list(reversed(xs))) == base,
        "append_zero_invariant": run_source(source, xs + [0]) == base,
        "duplicate_doubles": run_source(source, xs + xs) == 2 * base,
    }
    return {"checks": checks, "all_properties": all(checks.values()),
            "actual_on_anchor": base, "expected_on_anchor": 2, "anchor_passed": base == 2}


def hypothetical_false_accept_risk(false_accept_rate: float, wrong_candidates: int) -> float:
    """假设独立、同分布的错误候选；至少一个被错误放行，不等于最终一定选错。"""
    require(type(false_accept_rate) in (int, float) and math.isfinite(false_accept_rate)
            and 0 <= false_accept_rate <= 1, "误放概率应在[0,1]")
    require(type(wrong_candidates) is int and 0 <= wrong_candidates <= 100000,
            "候选数量需为有界非负整数")
    return 1 - (1 - false_accept_rate) ** wrong_candidates


def mechanism_experiments() -> dict:
    target = {"expected": "14"}
    examples = []
    for text in ["14", "140", "28/2", "-14", "14.0", "14 或 140"]:
        try:
            strict = verify_exact(target, {"answer": text})["passed"]
        except LessonError:
            strict = False
        examples.append({"answer": text, "substring_accepts": "14" in text,
                         "exact_accepts": strict})
    wrong = {name: verify_code(next(t for t in make_tasks() if t.get("oracle") == name),
                              {"code": source}) for name, source in WRONG_CODE.items()}
    # 过程与答案分开：最终值正确，不意味着一条可见算式也正确。
    visible_step = {"left": "2", "right": "2", "claimed_sum": "5"}
    step_ok = (parse_rational(visible_step["left"]) + parse_rational(visible_step["right"])
               == parse_rational(visible_step["claimed_sum"]))
    witness_task = next(t for t in make_tasks() if t["id"] == "subset_positive")
    return {
        "numeric_audit": examples,
        "decimal": {"float_sum": 0.1 + 0.2, "exact_sum": str(Fraction("0.1") + Fraction("0.2")),
                    "fraction_from_float_equals_string": Fraction(0.3) == Fraction("0.3"),
                    "exact_rounded_third": parse_rational("0.333333") == Fraction(1, 3),
                    "declared_atol_1e6_accepts": close_rational("0.333333", "1/3", atol="0.000001", rtol="0")},
        "quantity": {"equivalent": verify_quantity({"expected": "90", "unit": "min"},
                                                   {"answer": "1.5", "unit": "h"}),
                     "wrong_dimension": verify_quantity({"expected": "90", "unit": "min"},
                                                         {"answer": "90", "unit": "m"})},
        "multiple_witnesses": [verify_witness(witness_task, {"items": x})
                               for x in [[1, 4, 7], [3, 4, 5], [7, 4, 1], [4, 4, 4]]],
        "code_public_vs_evaluation": wrong,
        "metamorphic_zero_program": metamorphic_report("def solve(numbers):\n    return 0\n"),
        "outcome_vs_process": {"visible_step": visible_step, "step_is_correct": step_ok,
                               "final_only_grader_accepts_4": verify_exact({"expected": "4"}, {"answer": "4"})["passed"],
                               "not_a_learned_process_verifier": True},
        "selection_warning": "评测参考答案可离线计分，不能冒充部署时可用的选择器。",
        "hypothetical_false_accept_risk": {"assumed_false_accept_rate": 0.01,
            "assumed_independent_wrong_candidates": 100,
            "at_least_one_accepted": hypothetical_false_accept_risk(0.01, 100),
            "not_an_empirical_model_measurement": True},
    }

In [8]:
def quantity_ex_reference(answer: str, unit: str, expected: str, expected_unit: str) -> bool:
    actual_dim, actual_value = normalize_quantity(answer, unit)
    expected_dim, expected_value = normalize_quantity(expected, expected_unit)
    return actual_dim == expected_dim and actual_value == expected_value


def witness_ex_reference(items: list[int], allowed: list[int], count: int, target: int) -> bool:
    require(type(items) is list and all(type(x) is int for x in items), "需要整数列表")
    return (len(items) == count and len(set(items)) == count
            and all(x in allowed for x in items) and sum(items) == target)


def aggregate_ex_reference(planned: list[str], received: dict[str, bool]) -> dict:
    require(len(planned) > 0 and len(set(planned)) == len(planned), "计划为空或重复")
    require(set(received) <= set(planned), "未知ID")
    require(all(type(v) is bool for v in received.values()), "结果必须为bool")
    passed = sum(received.get(key) is True for key in planned)
    return {"planned": len(planned), "received": len(received), "passed": passed,
            "rate": passed / len(planned), "missing": sorted(set(planned) - set(received))}


def check_quantity_ex(fn) -> None:
    require(fn("1.5", "h", "90", "min") is True, "等价时间换算失败")
    require(fn("75", "cm", "3/4", "m") is True, "等价长度失败")
    require(fn("75", "m", "75", "cm") is False, "漏掉单位倍数")
    require(fn("90", "m", "90", "min") is False, "漏掉物理量类别")
    require(fn("750", "fen", "7.5", "CNY") is True, "金额精确换算失败")


def check_witness_ex(fn) -> None:
    allowed = list(range(1, 9))
    require(fn([1, 4, 7], allowed, 3, 12) is True, "第一种合法见证失败")
    require(fn([3, 4, 5], allowed, 3, 12) is True, "不应只接受一种答案")
    require(fn([7, 4, 1], allowed, 3, 12) is True, "顺序应无关")
    require(fn([4, 4, 4], allowed, 3, 12) is False, "漏掉重复约束")
    require(fn([0, 5, 7], allowed, 3, 12) is False, "漏掉取值范围")
    require(fn([4, 8], allowed, 3, 12) is False, "漏掉元素数量")


def check_aggregate_ex(fn) -> None:
    result = fn(["a", "b", "c"], {"a": True, "b": False})
    require(result == {"planned": 3, "received": 2, "passed": 1, "rate": 1 / 3,
                       "missing": ["c"]}, "缺失项被移出分母")
    for planned, received in [(["a", "a"], {}), (["a"], {"x": True}), (["a"], {"a": 1})]:
        rejected = False
        try: fn(planned, received)
        except (ValueError, TypeError): rejected = True
        require(rejected, "非法计划或非bool结果未被拒绝")

In [9]:
def run_checks() -> dict:
    names = []
    def check(name, condition):
        require(bool(condition), "检查失败：" + name)
        names.append(name)
    def rejects(name, fn):
        try:
            fn()
        except (LessonError, ValueError, TypeError):
            names.append(name)
            return
        raise LessonError("未拒绝：" + name)
    tasks = make_tasks()
    test = {t["id"]: t for t in tasks if t["split"] == "test"}
    check("4道开发+12道测试", len(tasks) == 16 and len(test) == 12)
    check("任务ID唯一", len({t["id"] for t in tasks}) == len(tasks))
    check("精确小数", parse_rational("0.1") + parse_rational("0.2") == Fraction(3, 10))
    check("等价分数", parse_rational("2/4") == parse_rational("0.5"))
    check("保留负号", parse_rational("-14") != 14)
    check("外侧空白允许", parse_rational("  +14.00\n") == 14)
    for name, value in [("拒绝浮点输入", .3), ("拒绝布尔答案", True), ("拒绝混合答案", "14 or 15"),
                        ("拒绝嵌入句子", "答案是14"), ("拒绝零分母", "1/0"), ("拒绝NaN", "NaN"),
                        ("拒绝指数", "1e3"), ("拒绝全角数字", "１４"), ("限制数字位数", "9" * 25),
                        ("不执行算式", "1+2"), ("不自动解析百分号", "50%")]:
        rejects(name, lambda v=value: parse_rational(v))
    check("精确不接受近似", parse_rational("0.333333") != Fraction(1, 3))
    check("显式绝对容差", close_rational("0.333333", "1/3", atol="0.000001", rtol="0"))
    rejects("拒绝负容差", lambda: close_rational("1", "1", atol="-1", rtol="0"))
    check("零附近绝对容差", close_rational("0.0001", "0", atol="0.001", rtol="0"))
    for name, raw in [("重复键", '{"answer":"1","answer":"2"}'),
                      ("非有限JSON", '{"answer":NaN}'), ("代码围栏不修复", '```json\n{}\n```'),
                      ("尾随解释", '{} done'), ("多对象", '{}{}'), ("非法Unicode", '"\\ud800"')]:
        rejects(name, lambda r=raw: strict_json(r))
    check("精确字段schema", verify_exact(test["fraction_sum"], {"answer": "0.5"})["passed"])
    rejects("额外reward字段", lambda: verify_exact(test["fraction_sum"], {"answer": "0.5", "reward": 1}))
    check_quantity_ex(quantity_ex_reference); names.append("单位练习参考实现")
    rejects("未知单位", lambda: normalize_quantity("90", "minutesXYZ"))
    check("错单位明确失败", not verify_quantity(test["duration"], {"answer": "90", "unit": "m"})["passed"])
    check("金额保留小数", verify_quantity(test["price"], {"answer": "7.50", "unit": "元"})["passed"])
    check_witness_ex(witness_ex_reference); names.append("见证练习参考实现")
    check("重复见证拒绝", not verify_witness(test["subset_positive"], {"items": [4, 4, 4]})["passed"])
    rejects("bool见证拒绝", lambda: verify_witness(test["subset_positive"], {"items": [True, 4, 7]}))
    check("合法不同见证", verify_witness(test["subset_positive"], {"items": [3, 4, 5]})["passed"])
    for key, expected in [("sum_abs", 6), ("count_nonzero", 2), ("max_or_zero", 4), ("sum_nonnegative", 4)]:
        check("解释参考函数_" + key, run_source(CORRECT_CODE[key], [-2, 0, 4]) == expected)
    check("全负最大值", run_source(CORRECT_CODE["max_or_zero"], [-12, -3]) == -3)
    check("空输入", run_source(CORRECT_CODE["max_or_zero"], []) == 0)
    xs = [-1, 0, 2]; before = xs[:]
    run_source(CORRECT_CODE["count_nonzero"], xs)
    check("不修改调用输入", xs == before)
    for label, src in {
        "import": "import os\ndef solve(numbers):\n return 0\n",
        "call": "def solve(numbers):\n return sum(numbers)\n",
        "attribute": "def solve(numbers):\n return numbers.__class__\n",
        "while": "def solve(numbers):\n while True:\n  return 0\n",
        "index": "def solve(numbers):\n return numbers[0]\n",
        "decorator": "@x\ndef solve(numbers):\n return 0\n",
        "default_argument": "def solve(numbers=[]):\n return 0\n",
        "bool_return": "def solve(numbers):\n return True\n",
        "replace_input": "def solve(numbers):\n numbers = 1\n return 0\n",
        "power": "def solve(numbers):\n return 2**10\n",
    }.items():
        rejects("禁止语法_" + label, lambda s=src: parse_program(s))
    rejects("未初始化变量", lambda: run_source("def solve(numbers):\n return a\n", []))
    rejects("必须返回", lambda: run_source("def solve(numbers):\n x = 1\n", []))
    rejects("除零运行错误", lambda: run_source("def solve(numbers):\n return 1//0\n", []))
    rejects("输入拒绝bool", lambda: run_source(CORRECT_CODE["sum_abs"], [True]))
    rejects("输入长度预算", lambda: run_source(CORRECT_CODE["sum_abs"], [1] * 17))
    rejects("中间整数范围", lambda: run_source("def solve(numbers):\n return 1000000000 * 1000000000\n", []))
    rejects("步骤预算", lambda: run_source(CORRECT_CODE["sum_abs"], [1, 2], step_limit=1))
    for name, src in WRONG_CODE.items():
        task = test["code_" + name]
        report = verify_code(task, {"code": src})
        check("公开样例通过_" + name, report["public"]["passed"])
        check("评测样例拒绝_" + name, not report["evaluation"]["passed"] and not report["passed"])
    for task in tasks:
        if task["kind"] == "code":
            public_inputs = {tuple(c["input"]) for c in task["public_cases"]}
            check("测试不重叠_" + task["id"],
                  not public_inputs & {tuple(c["input"]) for c in task["evaluation_cases"]})
    reference = score_records(fixture_records("reference"))
    check("参考夹具12题全通", reference["summary"]["verified_passed"] == 12)
    overfit = score_records(fixture_records("public_overfit"))
    check("公开过拟合不能全通", overfit["summary"]["verified_passed"] == 9)
    mixed = score_records(fixture_records("mixed"))
    check("固定分母12", mixed["summary"]["planned"] == 12 and mixed["summary"]["received"] == 11)
    check("混合夹具5题通过", mixed["summary"]["verified_passed"] == 5
          and mixed["summary"]["success_rate"] == 5 / 12)
    check("错误缺失截断分开", all(mixed["summary"]["stages"][s] == 1
                                for s in ["generation_error", "incomplete_generation", "missing"]))
    cut = candidate("fraction_sum", {"answer": "1/2"}, stop_reason="length")
    check("截断正确文本不执行", grade_record(test["fraction_sum"], cut)["answer_ok"] is None)
    ref_rows = fixture_records("reference")
    rejects("重复响应不得择优", lambda: score_records(ref_rows + [ref_rows[0]]))
    bad = copy.deepcopy(ref_rows); bad[0]["task_id"] = "unknown_task"
    rejects("未知响应", lambda: score_records(bad))
    bad = copy.deepcopy(ref_rows); bad[0]["reward"] = 1
    rejects("响应不允许自报得分", lambda: score_records(bad))
    check("空记录保留分母", score_records([])["summary"]["planned"] == 12)
    changed = copy.deepcopy(make_plan()); changed["dataset_sha256"] = "fake"
    rejects("计划不匹配", lambda: score_records(ref_rows, plan=changed))
    for task in tasks:
        request = public_request(task)
        altered = copy.deepcopy(task)
        if "expected" in altered: altered["expected"] = "999999"
        if "evaluation_cases" in altered: altered["evaluation_cases"] = []
        check("答案不流入请求_" + task["id"], request == public_request(altered))
    check("属性检查不能代替答案", metamorphic_report("def solve(numbers):\n return 0\n")["all_properties"])
    check("属性常零反例确实错误", not metamorphic_report("def solve(numbers):\n return 0\n")["anchor_passed"])
    check_aggregate_ex(aggregate_ex_reference); names.append("固定分母练习参考实现")
    check("假设零误放风险", hypothetical_false_accept_risk(0, 100) == 0)
    check("100次假设误放风险", abs(hypothetical_false_accept_risk(.01, 100) - .6339676587267709) < 1e-12)
    rejects("误放率拒绝bool", lambda: hypothetical_false_accept_risk(True, 2))
    malformed = copy.deepcopy(ref_rows[0]); malformed["status"] = []
    rejects("状态类型严格检查", lambda: validate_record(malformed))
    check("dev独立可运行", score_records(fixture_records("reference", "dev"), "dev")["summary"]["verified_passed"] == 4)
    return {"passed": len(names), "checks": names,
            "scope": "standard_library_math_and_bounded_interpreter_not_model_or_sandbox_certification"}

In [10]:
@contextmanager
def replaced_global(name: str, value):
    old = globals()[name]
    globals()[name] = value
    try:
        yield
    finally:
        globals()[name] = old


def mutation_checks() -> dict:
    original_quantity = verify_quantity
    original_code = verify_code
    original_summary = summarize
    original_grade = grade_record
    def ignore_units(task, payload):
        result = original_quantity(task, payload)
        result["passed"] = parse_rational(payload["answer"]) == Fraction(task["expected"])
        return result
    def public_only(task, payload):
        result = original_code(task, payload)
        result["passed"] = result["public"]["passed"]
        return result
    def remove_missing(rows):
        return original_summary([r for r in rows if r["stage"] != "missing"])
    def accept_truncated(task, record):
        if record is not None and record["status"] == "ok":
            record = dict(record, stop_reason="eos")
        return original_grade(task, record)
    def sum_only(task, payload):
        return {"passed": sum(payload["items"]) == task["target_sum"]}
    mutations = [
        ("先float再Fraction", "parse_rational", lambda s: Fraction(float(s)) if "/" not in s else Fraction(s)),
        ("忽略单位", "verify_quantity", ignore_units),
        ("只检查见证总和", "verify_witness", sum_only),
        ("只用公开代码样例", "verify_code", public_only),
        ("删除缺失分母", "summarize", remove_missing),
        ("接受截断输出", "grade_record", accept_truncated),
    ]
    results = []
    for label, name, faulty in mutations:
        caught, reason = False, ""
        with replaced_global(name, faulty):
            try:
                run_checks()
            except LessonError as exc:
                caught, reason = True, str(exc)
        require(caught, "故障未被检测：" + label)
        results.append({"mutation": label, "caught": caught, "first_failed_check": reason})
    return {"detected": sum(r["caught"] for r in results), "total": len(results), "results": results}

In [11]:
def save_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def save_jsonl(path: Path, values: list[dict]) -> None:
    path.write_text("".join(canonical(v) + "\n" for v in values), encoding="utf-8")


def read_jsonl(path: Path) -> list[dict]:
    require(path.is_file(), "输入JSONL不存在")
    require(path.stat().st_size <= MAX_FILE_BYTES, "输入文件过大")
    records = []
    with path.open("r", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            require(len(records) < MAX_RECORDS, "记录过多")
            require(line.strip() != "", f"第{line_number}行为空")
            records.append(strict_json(line))
    return records


def new_run(base: str | Path, label: str) -> Path:
    root = Path(base).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    run = root / f"{label}_{stamp}"
    run.mkdir(exist_ok=False)
    return run


def export_bundle(base="lesson_55_outputs", split="test") -> Path:
    out = new_run(base, "export")
    tasks = [t for t in make_tasks() if t["split"] == split]
    save_json(out / "plan.json", make_plan(split))
    save_jsonl(out / "requests.jsonl", [public_request(t) for t in tasks])
    save_jsonl(out / "responses_template.jsonl", [candidate(t["id"], "", status="error", stop_reason="error") for t in tasks])
    save_json(out / "contract.json", grader_contract())
    save_json(out / "manifest.json", {"mode": "export_only", "version": VERSION,
        "warning": "responses_template 全部为占位错误，需写入真实完整响应。不得把参考答案输入模型。"})
    return out


def write_score(out: Path, records: list[dict], split: str, source_mode: str) -> dict:
    plan = make_plan(split)
    report = score_records(records, split, plan)
    save_json(out / "plan.json", plan)
    save_json(out / "tasks_for_grader.json", [t for t in make_tasks() if t["split"] == split])
    save_jsonl(out / "raw.jsonl", records)
    save_json(out / "report.json", report)
    save_json(out / "manifest.json", {"mode": source_mode, "version": VERSION,
        "python": platform.python_version(), "record_count": len(records), "raw_sha256": digest(records),
        "not_a_model_benchmark_by_default": True, "no_candidate_python_execution": True,
        "scope_warning": "公开教学评分器。记录与模型来源未经身份认证；有限测试不是形式证明。"})
    return report


def default_experiment(base="lesson_55_outputs", *, include_checks=True) -> tuple[Path, dict]:
    out = new_run(base, "offline")
    checks = run_checks() if include_checks else {"skipped": True}
    mechanism = mechanism_experiments()
    reports = {}
    for name in ["reference", "public_overfit", "mixed"]:
        sub = out / name
        sub.mkdir()
        reports[name] = write_score(sub, fixture_records(name), "test", "handwritten_fixture")
    summary = {name: report["summary"] for name, report in reports.items()}
    save_json(out / "checks.json", checks)
    save_json(out / "mechanisms.json", mechanism)
    save_json(out / "comparison.json", summary)
    save_json(out / "mutations.json", mutation_checks())
    save_json(out / "sources.json", SOURCES)
    save_json(out / "manifest.json", {"version": VERSION, "mode": "handwritten_fixture_experiment",
        "python": platform.python_version(), "model_called": False, "gpu_used": False,
        "tests_are_public": True, "scope": "数学与程序评分器的机制验证，不是模型能力成绩"})
    return out, summary


def rescore_run(run: str | Path, base="lesson_55_outputs") -> Path:
    run = Path(run).expanduser().resolve()
    plan_path = run / "plan.json"
    require(plan_path.is_file() and plan_path.stat().st_size <= MAX_RESPONSE_BYTES, "缺少合法plan.json")
    plan = strict_json(plan_path.read_text(encoding="utf-8"))
    require(type(plan) is dict and type(plan.get("split")) is str, "plan 缺少 split")
    records = read_jsonl(run / "raw.jsonl")
    score_records(records, plan["split"], plan)  # 拒绝变更计划；不信任旧report中的passed
    out = new_run(base, "rescore")
    write_score(out, records, plan["split"], "rescored_saved_records")
    return out


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    mode = parser.add_mutually_exclusive_group()
    mode.add_argument("--checks-only", action="store_true")
    mode.add_argument("--math-only", action="store_true")
    mode.add_argument("--export-prompts", action="store_true")
    mode.add_argument("--grade", type=Path, help="评分自己保存的响应JSONL，不调用模型")
    mode.add_argument("--rescore", type=Path, help="重评分一个包含plan.json与raw.jsonl的子目录")
    parser.add_argument("--split", choices=["dev", "test"], default="test")
    parser.add_argument("--output-dir", default="lesson_55_outputs")
    args = parser.parse_args(argv)
    try:
        if args.checks_only:
            report = run_checks()
            print(f"固定检查通过：{report['passed']}项。不是模型推理成绩或通用代码沙箱认证。")
            return 0
        if args.math_only:
            out = new_run(args.output_dir, "math")
            save_json(out / "mechanisms.json", mechanism_experiments())
        elif args.export_prompts:
            out = export_bundle(args.output_dir, args.split)
        elif args.grade:
            records = read_jsonl(args.grade)
            # 在创建输出之前完成协议校验，避免把错误输入变成一份成功报告。
            score_records(records, args.split)
            out = new_run(args.output_dir, "grade")
            report = write_score(out, records, args.split, "user_supplied_unverified_origin")
            print(canonical(report["summary"]))
        elif args.rescore:
            out = rescore_run(args.rescore, args.output_dir)
        else:
            out, summary = default_experiment(args.output_dir)
            for name, value in summary.items():
                print(f"{name}: {value['verified_passed']}/{value['planned']} (手工夹具)")
        print("结果目录：", out)
        return 0
    except (LessonError, OSError, UnicodeError) as exc:
        print("失败：", str(exc), file=sys.stderr)
        return 2

## 1．先看契约，不从回答里猜测契约

一个任务同时包含问题、输出格式和检查规则。以下是只允许单个精确数值的例子：

```json
{"answer":"1/2"}
```

数学上等价的 `"0.5"`、`"2/4"` 应通过；`"1/2 或 1/3"` 不是一个确定答案。`"答案是1/2"` 在本课 JSON 契约下也是格式不合格，而不是被证明数学推理错误。

程序不会删除解释、补 JSON、寻找最后一段数字，或接受模型自报的 `reward=1`。

Python JSON 默认接受重复键和非有限数扩展，因此本课额外限制键唯一、大小、深度和数值范围。[Python JSON](https://docs.python.org/3.13/library/json.html)

In [12]:
TASKS = make_tasks()
TEST = {t["id"]: t for t in TASKS if t["split"] == "test"}
print("开发题：", sum(t["split"] == "dev" for t in TASKS))
print("测试题：", len(TEST))
print(json.dumps(public_request(TEST["fraction_sum"]), ensure_ascii=False, indent=2))
try:
    strict_json('{"answer":"1/2","answer":"0"}')
except LessonError as error:
    print("重复键被拒绝：", error)

开发题： 4
测试题： 12
{
  "task_id": "fraction_sum",
  "sample_id": "s0",
  "kind": "exact",
  "problem": "计算 1/3 + 1/6。",
  "response_rule": "仅输出JSON {\"answer\":\"数值\"}；数值是整数、普通十进制小数或分数的字符串，合计最多24个数字；保留正负号，不用指数、百分号、单位、解释或代码围栏。精确比较，不四舍五入。"
}
重复键被拒绝： 重复 JSON 键


## 2．子串匹配为什么不可信？

以正确值 14 为例。下面先比较一个故意写坏的评分器 `"14" in text`，再比较严格数值检查。

**运行前预测：`140`、`-14`、`28/2` 会分别发生什么？**

误放（false acceptance）会让错误结果看起来成功；误拒（false rejection）会让等价的正确结果得不到分。两种都要检查。

In [13]:
MECHANISMS = mechanism_experiments()
for row in MECHANISMS["numeric_audit"]:
    print(f"{row['answer']!r:14} 子串={str(row['substring_accepts']):5} 精确={row['exact_accepts']}")

'14'           子串=True  精确=True
'140'          子串=True  精确=False
'28/2'         子串=False 精确=True
'-14'          子串=True  精确=False
'14.0'         子串=True  精确=True
'14 或 140'     子串=True  精确=False


## 3．精确算术：从字符串构造 Fraction

浮点数通常不能精确表示全部十进制小数。`Fraction(0.3)` 会精确保存那个已经舍入过的二进制浮点值，而不是自动恢复 `3/10`；从字符串构造可以表达本课想要的精确十进制含义。[Python fractions](https://docs.python.org/3.13/library/fractions.html)

本课只允许**一个**整数、普通小数或分数，最多24个数字。明确拒绝指数、百分号、单位混在数值中、Unicode 数字替代和零分母。它不是通用数学表达式解释器。

注意：输入解析规则比 Fraction 本身更窄，属于预先公布的课程契约。

In [14]:
print("浮点加法：", 0.1 + 0.2)
print("字符串精确加法：", parse_rational("0.1") + parse_rational("0.2"))
print("2/4 与 0.5：", parse_rational("2/4") == parse_rational("0.5"))
print("Fraction(0.3)：", Fraction(0.3))
print("Fraction('0.3')：", Fraction("0.3"))
print("近似小数等于 1/3？", parse_rational("0.333333") == Fraction(1, 3))

浮点加法： 0.30000000000000004
字符串精确加法： 3/10
2/4 与 0.5： True
Fraction(0.3)： 5404319552844595/18014398509481984
Fraction('0.3')： 3/10
近似小数等于 1/3？ False


### 近似任务必须单独声明容差

对于允许数值近似的任务，可以预先定义：

\[
|\widehat y-y|\leq \mathrm{atol}+\mathrm{rtol}|y|
\]

这是**本课独立示例采用的公式**，不是声称与所有库的 `isclose` 规则相同。精确题仍然用等号，不偷偷调用 `round()` 或 `limit_denominator()`。

相对容差在目标为零时没有帮助，可能需要事先规定绝对容差。不能看完答案再把阈值放宽。

In [15]:
print("精确接受：", parse_rational("0.333333") == Fraction(1, 3))
print("预先允许 atol=0.000001：",
      close_rational("0.333333", "1/3", atol="0.000001", rtol="0"))
print("零附近：", close_rational("0.0001", "0", atol="0.001", rtol="0"))

精确接受： False
预先允许 atol=0.000001： True
零附近： True


## 4．数值对了，单位也可能错

1.5 小时与 90 分钟是同一段时间；90 米与90分钟则不是同一物理量。

本课先确定物理量类别，再使用有理数换算到基准单位。它不猜测 `m` 是分钟还是米：本课 `m` 固定表示米，分钟使用 `min`、`minute` 或“分钟”。

这是显式的小型单位表，不实现通用量纲代数、温度仿射换算或货币汇率。金额题只是固定单价的教学乘法。

In [16]:
for payload in [
    {"answer": "1.5", "unit": "h"},
    {"answer": "5400", "unit": "s"},
    {"answer": "90", "unit": "m"},
    {"answer": "1.5", "unit": "min"},
]:
    print(payload, "→", verify_quantity(TEST["duration"], payload))

{'answer': '1.5', 'unit': 'h'} → {'passed': True, 'dimension': 'time', 'canonical_value': '5400', 'reason': 'equal_quantity', 'scope': 'quantity_with_explicit_unit_table'}
{'answer': '5400', 'unit': 's'} → {'passed': True, 'dimension': 'time', 'canonical_value': '5400', 'reason': 'equal_quantity', 'scope': 'quantity_with_explicit_unit_table'}
{'answer': '90', 'unit': 'm'} → {'passed': False, 'dimension': 'length', 'canonical_value': '90', 'reason': 'wrong_dimension', 'scope': 'quantity_with_explicit_unit_table'}
{'answer': '1.5', 'unit': 'min'} → {'passed': False, 'dimension': 'time', 'canonical_value': '90', 'reason': 'wrong_quantity', 'scope': 'quantity_with_explicit_unit_table'}


## 5．有多个合法答案时：验证约束，不是背参考字符串

题目：从1到8选出三个互不相同的整数，使它们的和为12。

`[1,4,7]`、`[3,4,5]` 与 `[7,4,1]` 都可以。`[4,4,4]` 虽然和为12，却违反互不重复。

验证器逐项检查：**数量、范围、唯一性、总和**。只比较总和会误放，只与第一份参考解比较会误拒。

在这个形式化的小任务里，逐项约束足以决定见证是否成立；不是对任意自然语言问题都能做到这一点。

In [17]:
for items in [[1, 4, 7], [3, 4, 5], [7, 4, 1], [4, 4, 4], [0, 5, 7]]:
    result = verify_witness(TEST["subset_positive"], {"items": items})
    print(items, "→", result["passed"], result["checks"])

[1, 4, 7] → True {'count': True, 'membership': True, 'distinct': True, 'sum': True}
[3, 4, 5] → True {'count': True, 'membership': True, 'distinct': True, 'sum': True}
[7, 4, 1] → True {'count': True, 'membership': True, 'distinct': True, 'sum': True}
[4, 4, 4] → False {'count': True, 'membership': True, 'distinct': False, 'sum': True}
[0, 5, 7] → False {'count': True, 'membership': False, 'distinct': True, 'sum': True}


## 6．最终值正确，不等于推理过程经过认证

结果监督与过程监督是不同路线。过程监督研究评估中间步骤，不只是最终答案。[Let's Verify Step by Step](https://arxiv.org/abs/2305.20050)

本课只核对提交的最终值、约束和测试行为，不训练 PRM，不检查模型隐藏思考，也不把语言流畅度作为得分。

下面故意把一条**外显算式**写错，但单独提交正确最终答案。最终值评分可以通过，外显算式检查仍失败。它说明的是评分范围，不要求模型披露任何内部推理。

In [18]:
print(json.dumps(MECHANISMS["outcome_vs_process"], ensure_ascii=False, indent=2))

{
  "visible_step": {
    "left": "2",
    "right": "2",
    "claimed_sum": "5"
  },
  "step_is_correct": false,
  "final_only_grader_accepts_4": true,
  "not_a_learned_process_verifier": true
}


## 7．代码评分：宿主测试，不是模型自己说“通过了”

本课代码任务要求输出：

```json
{"code":"def solve(numbers):\n    ..."}
```

输入是最多16个整数的列表，返回一个整数。完整语言规则见请求；支持局部整数、比较、if、遍历 `numbers` 的 for。**不支持 import、调用、属性、索引、while 或任意Python**。

候选只进入有限解释器，宿主控制测试与预期。这里没有 `exec()`、没有读写工具，也不让候选修改测试。

Python 的 AST 只是语法树工具，本身不是资源隔离或安全保证。本课限制源码、节点、深度、解释步数和整数范围，但没有生产安全认证。[Python AST](https://docs.python.org/3.13/library/ast.html)

In [19]:
task = TEST["code_count_nonzero"]
print(task["problem"])
print("公开样例：", task["public_cases"])
print("额外评测用例：", len(task["evaluation_cases"]))
print("正确参考代码：\n", CORRECT_CODE["count_nonzero"])
print("[-2,0,3] 的输出：", run_source(CORRECT_CODE["count_nonzero"], [-2, 0, 3]))

返回非零整数的数量；负数也算非零；空列表返回0。
公开样例： [{'input': [], 'expected': 0}, {'input': [0, 0], 'expected': 0}, {'input': [1, 0, 2], 'expected': 2}]
额外评测用例： 161
正确参考代码：
 def solve(numbers):
    count = 0
    for value in numbers:
        if value != 0:
            count = count + 1
    return count

[-2,0,3] 的输出： 2


### 合法的完整 Python，也可能超出课堂子集

例如 `return sum(numbers)` 对一些任务是合理 Python，但它包含函数调用，本课解释器不支持。

这类结果报告为 `unsupported_code`，**不宣称模型数学逻辑错误**。主要任务完成率仍按事先公布的语言契约计分。

扩展到真实项目时需要合适的隔离执行环境，而不是删掉本课限制后直接运行。

In [20]:
try:
    run_source("def solve(numbers):\n    return sum(numbers)\n", [1, 2])
except UnsupportedCode as error:
    print("超出语言子集：", error)

try:
    run_source(CORRECT_CODE["sum_abs"], [1, 2], step_limit=1)
except RuntimeLimit as error:
    print("解释步骤上限：", error)

超出语言子集： 不支持的语法：Call
解释步骤上限： step_budget_exhausted


## 8．公开样例全通过，为什么仍然失败？

三个预写的错误程序：

| 任务 | 故障 |
|---|---|
| 非零计数 | 只统计正数，遗漏负数 |
| 最大值，空输入才返回零 | 从零开始取最大值，全负输入错误 |
| 非负整数求和 | 偷加 `value < 10` 的限制 |

三道题各有3个公开样例。额外测试由固定的短列表网格与边界输入构成，不与公开样例重复。

**测试是在生成前确定的，不根据候选答案动态放宽。** 本课运行后会打印全部测试明细，所以它是一套公开回归材料，不能持续称作盲测。

In [21]:
for name, result in MECHANISMS["code_public_vs_evaluation"].items():
    p, e = result["public"], result["evaluation"]
    first_failure = next(c for c in e["cases"] if not c["passed"])
    print(name, f"公开{p['passed_count']}/{p['total']}",
          f"评测{e['passed_count']}/{e['total']}", "整题通过：", result["passed"])
    print("第一条额外失败：", first_failure)

count_nonzero 公开3/3 评测40/161 整题通过： False
第一条额外失败： {'input': [-2], 'expected': 1, 'actual': 0, 'error': None, 'steps': 10, 'passed': False}
max_or_zero 公开3/3 评测145/162 整题通过： False
第一条额外失败： {'input': [-2], 'expected': -2, 'actual': 0, 'error': None, 'steps': 14, 'passed': False}
sum_nonnegative 公开3/3 评测157/161 整题通过： False
第一条额外失败： {'input': [10, -5, 20], 'expected': 30, 'actual': 0, 'error': None, 'steps': 29, 'passed': False}


### 有限测试不是形式证明

一个候选通过全部当前测试，只能说“通过本套测试”。不能宣称所有合法输入、任意资源条件与安全行为都得到证明。

代码的一个成功单位是**整道任务通过全部约定用例**，不是把通过156个、失败1个也写成“基本修好了”。同时保留逐用例结果，便于定位问题。

官方 HumanEval 的执行路径也把函数执行与测试结果作为检查对象，并明确警告执行不可信代码的风险。[HumanEval](https://github.com/openai/human-eval/blob/master/human_eval/execution.py)

## 9．性质测试有帮助，但也有盲点

对“非零计数”，反转列表不应改变结果；添加零不应改变结果；列表复制一遍，计数应该翻倍。

下面的坏程序永远返回0。这三条关系全部成立，但它仍然没有完成任务。

**性质测试可以增加覆盖，不能不加思考地代替带真值的测试。**

In [22]:
zero_program = "def solve(numbers):\n    return 0\n"
print(json.dumps(metamorphic_report(zero_program), ensure_ascii=False, indent=2))

{
  "checks": {
    "reverse_invariant": true,
    "append_zero_invariant": true,
    "duplicate_doubles": true
  },
  "all_properties": true,
  "actual_on_anchor": 0,
  "expected_on_anchor": 2,
  "anchor_passed": false
}


## 10．把生成状态与答案状态分开

每条外部响应采用：

```json
{"task_id":"fraction_sum","sample_id":"s0","status":"ok","stop_reason":"eos","output":"{\"answer\":\"1/2\"}"}
```

`output` 是原始响应字符串，不是已经计算好的分数。本课不接受自报 `correct=true` 或 `reward=1`。

本课约定：只有 `status=ok` 且 `stop_reason=eos` 才进入答案检查。实际调用其他接口时，适配器需要根据真实完成状态映射，不能随便填写 `eos`。

截断、错误与缺失都保留，但不把它们混称为“答案已经被证明错误”。本课一题一个 `s0`，同一个位置出现两条响应会拒绝，不暗中取较好答案。

In [23]:
for record in [
    candidate("fraction_sum", {"answer":"1/2"}),
    candidate("fraction_sum", {"answer":"1/2"}, stop_reason="length"),
    candidate("fraction_sum", "", status="error", stop_reason="error"),
    None,
]:
    result = grade_record(TEST["fraction_sum"], record)
    print(result["stage"], "answer_ok=", result["answer_ok"], "strict=", result["strict_success"])

passed answer_ok= True strict= True
incomplete_generation answer_ok= None strict= False
generation_error answer_ok= None strict= False
missing answer_ok= None strict= False


## 11．固定题集：先检查评分器，再报告模型结果

本课有4道开发题、12道测试题，测试题每类3道：精确数值、带单位数值、构造见证、代码。

这里用三组**手工夹具**验证程序：

* `reference`：参考答案与正确参考代码。
* `public_overfit`：数学答案正确，但三个代码只通过公开样例。
* `mixed`：混入错误答案、错误单位、截断、运行失败与一条缺失记录。

它们不是模型名称，不允许把参考夹具的100%写成“本模型100%”。

In [24]:
REPORTS = {name: score_records(fixture_records(name))
           for name in ["reference", "public_overfit", "mixed"]}
for name, report in REPORTS.items():
    s = report["summary"]
    print(name, f"通过{s['verified_passed']}/{s['planned']}",
          "收到记录：", s["received"], "阶段：", s["stages"])

reference 通过12/12 收到记录： 12 阶段： {'passed': 12}
public_overfit 通过9/12 收到记录： 12 阶段： {'passed': 9, 'wrong_result': 3}
mixed 通过5/12 收到记录： 11 阶段： {'generation_error': 1, 'incomplete_generation': 1, 'missing': 1, 'passed': 5, 'wrong_result': 4}


### 混合夹具的分母为什么是12？

计划12题，只收到11条。其中一条运行错误、一条截断；正常完成的9条中，5条通过、4条答案错误。

主要成功率是：

\[
5/12\approx41.67\%
\]

不能变成 `5/11`，更不能只对9条正常完成的记录报告 `5/9` 而不说明。

同时单独报告收到的数量、格式通过数、每类成功率和失败阶段，避免把基础设施问题伪装成逻辑能力下降。

In [25]:
for row in REPORTS["mixed"]["rows"]:
    print(f"{row['task_id']:22} {row['stage']:24} {row['strict_success']}")
print("分类结果：", json.dumps(REPORTS["mixed"]["summary"]["categories"], ensure_ascii=False))

fraction_sum           passed                   True
signed_arithmetic      wrong_result             False
decimal_sum            wrong_result             False
duration               passed                   True
length                 wrong_result             False
price                  passed                   True
subset_positive        passed                   True
subset_signed          incomplete_generation    False
subset_primes          missing                  False
code_count_nonzero     wrong_result             False
code_max_or_zero       passed                   True
code_sum_nonnegative   generation_error         False
分类结果： {"code": {"planned": 3, "passed": 1, "rate": 0.3333333333333333}, "exact": {"planned": 3, "passed": 1, "rate": 0.3333333333333333}, "quantity": {"planned": 3, "passed": 2, "rate": 0.6666666666666666}, "witness": {"planned": 3, "passed": 1, "rate": 0.3333333333333333}}


## 12．模型请求不能携带评分答案

公共请求由白名单字段构造：题目、输出要求、代码公开样例。精确答案、代码评测用例、参考函数都留在评分侧。

下面只改参考答案，模型请求应完全不变。评分计划的数据指纹则应该改变。这项检查验证的是本地数据流，不证明预训练数据没有污染。

下一课进行候选选择时还要注意：**数学题的隐藏标准答案可用于离线评测，不一定是在部署时可用的验证器。** 用标准答案挑最优候选属于 oracle 分析，不应冒充实际选择器。

In [26]:
original = TEST["fraction_sum"]
altered = copy.deepcopy(original)
altered["expected"] = "999"
print("公共请求不变：", public_request(original) == public_request(altered))
print("评分题目摘要改变：", digest(original) != digest(altered))
code_request = public_request(TEST["code_count_nonzero"])
print("代码请求字段：", sorted(code_request))
print("含评测用例字段：", "evaluation_cases" in code_request)

公共请求不变： True
评分题目摘要改变： True
代码请求字段： ['kind', 'problem', 'public_examples', 'response_rule', 'sample_id', 'task_id']
含评测用例字段： False


## 13．为什么多采样前必须审计评分器？

假设每个错误候选独立地有1%概率被误放，那么100个错误候选中至少一个被误放的概率是：

\[
1-(1-0.01)^{100}\approx63.40\%
\]

这是独立同分布假设下的数学示例，不是本课测得的模型错误率，也不等于最终一定选中那个错误候选。

它说明：**增加候选数不仅增加找到好答案的机会，也可能增加利用评分漏洞的机会。** 先修评分器，再谈 best-of-N。

In [27]:
for n in [1, 10, 50, 100]:
    print(f"假设错误候选{n:3}个，至少一次误放：{hypothetical_false_accept_risk(.01, n):.2%}")

假设错误候选  1个，至少一次误放：1.00%
假设错误候选 10个，至少一次误放：9.56%
假设错误候选 50个，至少一次误放：39.50%
假设错误候选100个，至少一次误放：63.40%


## 14．不相信旧报告：保留原始响应并重新评分

默认运行创建新的时间戳目录，不覆盖旧记录。每个夹具目录保存：

| 文件 | 作用 |
|---|---|
| `plan.json` | split、题集指纹、规则版本与计划位置 |
| `tasks_for_grader.json` | 公开教学题目及宿主侧答案／测试，不能发给模型 |
| `raw.jsonl` | 原始响应，不把分数混入响应 |
| `report.json` | 分项检查、用例明细与固定分母汇总 |
| `manifest.json` | 运行环境、来源声明与验证边界 |

`rescore` 用当前内置题目与原始响应重算，先核对计划，不相信旧报告中的 `passed`。哈希用于发现错配，不是数字签名或来源认证。

In [28]:
RUN_OUTPUT, RUN_SUMMARY = default_experiment("lesson_55_outputs")
print("本次结果目录：", RUN_OUTPUT)
print("固定检查：", json.loads((RUN_OUTPUT / "checks.json").read_text())["passed"])
print("故障检出：", json.loads((RUN_OUTPUT / "mutations.json").read_text())["detected"])
RESCORED = rescore_run(RUN_OUTPUT / "mixed", "lesson_55_outputs")
a = json.loads((RUN_OUTPUT / "mixed" / "report.json").read_text())
b = json.loads((RESCORED / "report.json").read_text())
print("新目录重评分完全相同：", a == b)

本次结果目录： /mnt/data/_lesson55_validation/notebook/lesson_55_outputs/offline_20261102T135903_575467Z
固定检查： 106
故障检出： 6
新目录重评分完全相同： True


### 生成自己的模型响应，再交给评分器

本课不重复下载模型，也不提供伪造的模型回答。先导出请求，沿用你已经调通的模型脚本生成结果，最后按约定保存 JSONL。

终端命令：

```bash
python lesson_55_reasoning_tasks_and_verifiable_graders.py --export-prompts --split dev
python lesson_55_reasoning_tasks_and_verifiable_graders.py --grade my_responses.jsonl --split dev
```

开发协议确定后再用 `--split test`。每个题目只保留一次尝试；研究多次采样将在下一课显式扩展计划。

导出的 `responses_template.jsonl` 全部标记为错误占位，不是已取得的回答。请改写为真实状态和原始输出，不要复制本课参考答案作为模型成绩。

下面单元只做导出，不调用模型。

In [29]:
EXPORTED = export_bundle("lesson_55_outputs", split="dev")
print("开发请求导出到：", EXPORTED)
print((EXPORTED / "requests.jsonl").read_text(encoding="utf-8").splitlines()[0])
print("响应占位：", (EXPORTED / "responses_template.jsonl").read_text().splitlines()[0])

开发请求导出到： /mnt/data/_lesson55_validation/notebook/lesson_55_outputs/export_20261102T135903_814927Z
{"kind":"exact","problem":"计算 3/4 + 1/8。","response_rule":"仅输出JSON {\"answer\":\"数值\"}；数值是整数、普通十进制小数或分数的字符串，合计最多24个数字；保留正负号，不用指数、百分号、单位、解释或代码围栏。精确比较，不四舍五入。","sample_id":"s0","task_id":"dev_fraction"}
响应占位： {"output":"","sample_id":"s0","status":"error","stop_reason":"error","task_id":"dev_fraction"}


## 15．故障注入：测试真的能抓住实现错误吗？

固定检查不仅检查“正常示例跑通”。下面在临时替换函数后再次运行原测试，检查六种错误是否被发现，然后恢复原实现。

它不修改磁盘上的评分器，也不执行候选任意代码。

In [30]:
faults = mutation_checks()
for result in faults["results"]:
    print(result["mutation"], "→", result["caught"], result["first_failed_check"])
print("检测数量：", faults["detected"], "/", faults["total"])

先float再Fraction → True 检查失败：精确小数
忽略单位 → True 检查失败：错单位明确失败
只检查见证总和 → True 检查失败：重复见证拒绝
只用公开代码样例 → True 检查失败：评测样例拒绝_count_nonzero
删除缺失分母 → True 检查失败：固定分母12
接受截断输出 → True 检查失败：混合夹具5题通过
检测数量： 6 / 6


# 独立练习

先自行编写，再打开检查开关。`NotImplementedError` 表示尚未完成，不要删掉检查来获得“全部通过”。参考答案位于后方，使用独立函数名，不覆盖你的实现。

## 练习 A：带单位的精确比较

实现 `quantity_ex(answer, unit, expected, expected_unit)`。

要求：数值通过 `parse_rational` 精确解析；使用 `UNITS` 的类别与倍数；等价单位通过，错误维度拒绝；不修改全局单位表。

In [31]:
RUN_EXERCISE_A = False

def quantity_ex(answer, unit, expected, expected_unit):
    raise NotImplementedError("请实现：类别匹配＋有理数精确换算")

if RUN_EXERCISE_A:
    check_quantity_ex(quantity_ex)
    print("练习 A 通过")

## 练习 B：验证见证，而不是匹配某一个答案

实现 `witness_ex(items, allowed, count, target)`，核对数量、范围、唯一性和总和。

请特别检查 `[4,4,4]`、`[0,5,7]`，以及合法答案的不同顺序。只包含 bool 的所谓整数见证不能接受。

In [32]:
RUN_EXERCISE_B = False

def witness_ex(items, allowed, count, target):
    raise NotImplementedError("请实现四项约束")

if RUN_EXERCISE_B:
    check_witness_ex(witness_ex)
    print("练习 B 通过")

## 练习 C：固定计划分母

实现 `aggregate_ex(planned, received)`。

`planned` 是题目 ID 列表，`received` 是 `ID -> bool` 的结果字典。返回计划数、收到数、通过数、通过比例和排序后的缺失 ID。

拒绝重复计划、未知响应 ID 与 `1/0` 冒充布尔值。不要对缺失记录补“已证明错误”，只是不给它计成功。

In [33]:
RUN_EXERCISE_C = False

def aggregate_ex(planned, received):
    raise NotImplementedError("请使用计划数量作分母")

if RUN_EXERCISE_C:
    check_aggregate_ex(aggregate_ex)
    print("练习 C 通过")

# 参考答案：完成练习后再核对

这些函数使用新的名称。前面的学生函数保持不变；参考答案检查通过，不代表你已经完成独立练习。

In [34]:
def quantity_answer(answer, unit, expected, expected_unit):
    require(unit in UNITS and expected_unit in UNITS, "未知单位")
    actual_dimension, actual_scale = UNITS[unit]
    expected_dimension, expected_scale = UNITS[expected_unit]
    actual_value = parse_rational(answer) * Fraction(actual_scale)
    expected_value = parse_rational(expected) * Fraction(expected_scale)
    return actual_dimension == expected_dimension and actual_value == expected_value

check_quantity_ex(quantity_answer)
print("参考 A 通过")

参考 A 通过


In [35]:
def witness_answer(items, allowed, count, target):
    require(type(items) is list and all(type(x) is int for x in items), "需要整数列表")
    if len(items) != count:
        return False
    if len(set(items)) != count:
        return False
    if any(value not in allowed for value in items):
        return False
    return sum(items) == target

check_witness_ex(witness_answer)
print("参考 B 通过")

参考 B 通过


In [36]:
def aggregate_answer(planned, received):
    require(len(planned) > 0 and len(set(planned)) == len(planned), "计划为空或重复")
    require(set(received) <= set(planned), "未知响应 ID")
    require(all(type(value) is bool for value in received.values()), "只接受 bool")
    passed = sum(received.get(key) is True for key in planned)
    return {
        "planned": len(planned),
        "received": len(received),
        "passed": passed,
        "rate": passed / len(planned),
        "missing": sorted(set(planned) - set(received)),
    }

check_aggregate_ex(aggregate_answer)
print("参考 C 通过")

参考 C 通过


## 终端命令备忘

```bash
# 只运行固定检查，不创建实验报告
python lesson_55_reasoning_tasks_and_verifiable_graders.py --checks-only

# 完整离线实验：三个手工夹具、机制、故障注入
python lesson_55_reasoning_tasks_and_verifiable_graders.py

# 只观察数学、代码评分反例
python lesson_55_reasoning_tasks_and_verifiable_graders.py --math-only

# 导出开发请求；不含精确答案或额外评测测试
python lesson_55_reasoning_tasks_and_verifiable_graders.py --export-prompts --split dev

# 使用真实保存的模型输出；一题一次，不自动挑最优
python lesson_55_reasoning_tasks_and_verifiable_graders.py --grade my_responses.jsonl --split dev

# 指向含 plan.json 和 raw.jsonl 的具体子目录
python lesson_55_reasoning_tasks_and_verifiable_graders.py --rescore lesson_55_outputs/实际offline目录/mixed
```

使用 `py` 或 `python3` 的环境，沿用你已跑通的命令。全课不需要GPU；没有tokens/s、显存或模型正确率数据。

## 验收与后续

能独立解释：

* 为什么 `28/2` 应当得到14分值的正确判定，但 `140` 不能？
* 为什么 90 分钟与1.5小时等价，90米却不行？
* 为什么约束题不应只接受一份参考解？
* 为什么3/3公开样例通过，仍可能整题失败？
* 为什么漏一条响应后，分母仍然是计划题目数？
* 哪些验证信息在真实部署时可能拿不到？

**本课建立的是评分底座，不是已经证明模型推理能力变强。** 完成练习后，把自己模型的真实输出导入、核对原始记录与报告，才得到该模型在此协议上的一次测量。

下一课：**第56课——多次采样、Self-consistency 与 Best-of-N**。固定模型和题目，区分候选覆盖、实际选择器表现和 oracle 上界，同时记录额外计算预算。

## 参考资料（原始资料／官方文档）

1. [Python Fraction：字符串与浮点构造的差异](https://docs.python.org/3.13/library/fractions.html)
2. [Python JSON：重复键、非有限数及默认行为](https://docs.python.org/3.13/library/json.html)
3. [Python AST：语法树接口与资源限制提醒](https://docs.python.org/3.13/library/ast.html)
4. [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168)
5. [Let's Verify Step by Step](https://arxiv.org/abs/2305.20050)
6. [HumanEval 执行代码中的安全警告](https://github.com/openai/human-eval/blob/master/human_eval/execution.py)

本课不调用上述项目的模型、数据集或执行器；教学题、有限解释器和夹具由本课自行构造。